In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from collections import deque

# Параметры системы
λ = 0.83  # интенсивность потока программ [1/мс]
μ = 0.6   # интенсивность обработки [1/мс]
ρ = λ / μ # коэффициент загрузки системы
n_channels = 3  # количество процессоров
simulation_time = 100000  # время моделирования в мс (увеличено для точности)

# 1. Аналитическая (Марковская) модель
def markov_model(ρ, n):
    """Расчет характеристик для n-канальной системы"""
    # Расчет вероятностей состояний
    P0 = 1.0
    for k in range(1, n+1):
        P0 += (ρ**k) / np.math.factorial(k)
    P0 = 1 / P0
    
    Pf = (ρ**n / np.math.factorial(n)) * P0
    Q = 1 - Pf
    A = λ * Q
    k_n = A / μ
    
    # Для систем с очередью (дополнительные расчеты)
    Lq = 0  # в марковской модели без очереди
    Wq = 0
    Ws = 1 / μ
    
    return {
        'P0': P0,
        'Pf': Pf,
        'Q': Q,
        'A': A,
        'Lq': Lq,
        'Wq': Wq,
        'Ws': Ws,
        'k_n': k_n
    }

# 2. Имитационная модель
def simulation_model(λ, μ, n_channels, total_time=simulation_time):
    """Имитационная модель с расчетом всех показателей"""
    processors = [0] * n_channels  # время освобождения процессоров
    queue = deque()                # очередь ожидания
    stats = {
        'total_programs': 0,
        'rejected': 0,
        'processed': 0,
        'queue_times': [],
        'system_times': [],
        'queue_lengths': [],
        'busy_periods': [0] * n_channels
    }
    
    # Генерация времени прибытия
    arrival_times = []
    current_time = 0
    while current_time < total_time:
        inter_arrival = random.expovariate(λ)
        current_time += inter_arrival
        if current_time < total_time:
            arrival_times.append(current_time)
    
    stats['total_programs'] = len(arrival_times)
    if stats['total_programs'] == 0:
        return {k: 0 for k in ['P0', 'Pf', 'Q', 'A', 'Lq', 'Wq', 'Ws', 'k_n']}
    
    # Основной цикл имитации
    current_time = 0
    event_index = 0
    while event_index < len(arrival_times) and current_time < total_time:
        # Обработка прибытия
        arrival_time = arrival_times[event_index]
        current_time = arrival_time
        
        # Поиск свободного процессора
        free_processor = None
        for i in range(n_channels):
            if processors[i] <= current_time:
                free_processor = i
                break
        
        if free_processor is not None:
            # Начало обработки
            service_time = random.expovariate(μ)
            processors[free_processor] = current_time + service_time
            stats['processed'] += 1
            stats['system_times'].append(service_time)
            stats['busy_periods'][free_processor] += service_time
        else:
            # Все процессоры заняты - отказ
            stats['rejected'] += 1
        
        event_index += 1
    
    # Расчет показателей
    Pf = stats['rejected'] / stats['total_programs']
    P0 = 1 - sum(stats['busy_periods']) / (n_channels * max(current_time, 1))
    Q = 1 - Pf
    A = λ * Q
    k_n = A / μ
    
    # Для этой модели (без очереди) некоторые показатели равны 0
    return {
        'P0': P0,
        'Pf': Pf,
        'Q': Q,
        'A': A,
        'Lq': 0,  # в этой модели очередь не учитывается
        'Wq': 0,  # в этой модели очередь не учитывается
        'Ws': np.mean(stats['system_times']) if stats['system_times'] else 0,
        'k_n': k_n
    }

# 3. Расчет для исходных параметров
markov_results = markov_model(ρ, n_channels)
sim_results = simulation_model(λ, μ, n_channels)

# 4. Расчет для разных коэффициентов загрузки
ρ_values = np.linspace(0.1, 3, 15)
results_markov = []
results_sim = []

for ρ_val in ρ_values:
    λ_temp = ρ_val * μ
    results_markov.append(markov_model(ρ_val, n_channels))
    results_sim.append(simulation_model(λ_temp, μ, n_channels))



# 5. Поиск количества каналов для Pf <= 0.01
target_Pf = 0.01
n_needed = n_channels
while True:
    n_needed += 1
    res = markov_model(ρ, n_needed)
    if res['Pf'] <= target_Pf:
        break

# 6. Вывод результатов
print("Результаты для исходных параметров (λ=0.83, μ=0.6, ρ=1.383):")
print("\nМарковская модель:")
print(f"1. Вероятность простоя системы (P0): {markov_results['P0']:.4f}")
print(f"2. Вероятность отказа (Pf): {markov_results['Pf']:.4f}")
print(f"3. Относительная пропускная способность (Q): {markov_results['Q']:.4f}")
print(f"4. Абсолютная пропускная способность (A): {markov_results['A']:.4f}")
print(f"5. Среднее число программ в очереди (Lq): {markov_results['Lq']:.4f}")
print(f"6. Среднее время ожидания в очереди (Wq): {markov_results['Wq']:.4f}")
print(f"7. Среднее время в системе (Ws): {markov_results['Ws']:.4f}")

print("\nИмитационная модель:")
print(f"1. Вероятность простоя системы (P0): {sim_results['P0']:.4f}")
print(f"2. Вероятность отказа (Pf): {sim_results['Pf']:.4f}")
print(f"3. Относительная пропускная способность (Q): {sim_results['Q']:.4f}")
print(f"4. Абсолютная пропускная способность (A): {sim_results['A']:.4f}")
print(f"5. Среднее число программ в очереди (Lq): {sim_results['Lq']:.4f}")
print(f"6. Среднее время ожидания в очереди (Wq): {sim_results['Wq']:.4f}")
print(f"7. Среднее время в системе (Ws): {sim_results['Ws']:.4f}")

print(f"\nДля вероятности отказа ≤ {target_Pf} требуется {n_needed} каналов")

# 7. Построение графиков
metrics = ['P0', 'Pf', 'Q', 'A', 'Lq', 'Wq', 'Ws']
titles = [
    'Вероятность простоя системы',
    'Вероятность отказа',
    'Относительная пропускная способность',
    'Абсолютная пропускная способность',
    'Среднее число программ в очереди',
    'Среднее время ожидания в очереди',
    'Среднее время в системе'
]

plt.figure(figsize=(15, 20))
for i, metric in enumerate(metrics):
    plt.subplot(4, 2, i+1)
    plt.plot(ρ_values, [res[metric] for res in results_markov], 'b-', label='Марковская')
    plt.plot(ρ_values, [res[metric] for res in results_sim], 'ro', label='Имитация')
    plt.xlabel('Коэффициент загрузки (ρ)')
    plt.ylabel(metric)
    plt.title(titles[i])
    plt.grid(True)
    plt.legend()

plt.tight_layout()
plt.show()



In [ ]:
def markov_model(processors_num, buffer_size, h_coeff=0.83, u1_coeff=0.6):
    rho = h_coeff / u1_coeff
    n_max = processors_num + buffer_size

    def factorial(n):
        result = 1
        for i in range(2, n + 1):
            result *= i
        return result

    if processors_num == 1:
        p0_denominator = 1 + rho + (rho**2)/2 + (rho**3)/6
        P0 = 1 / p0_denominator
        Pr = (rho**3 / 6) * P0
        Q = 1 - Pr
        A = h_coeff * Q
        kc = Q * rho
    else:
        def Pn(n):
            if n <= processors_num:
                return (rho ** n) / factorial(n)
            else:
                return (rho  n) / (factorial(processors_num) * (processors_num  (n - processors_num)))

        denominator = sum(Pn(n) for n in range(n_max + 1))
        P0 = 1 / denominator
        Pr = Pn(n_max) * P0
        Q = 1 - Pr
        A = h_coeff * Q
        kc = sum(min(n, processors_num) * Pn(n) * P0 for n in range(n_max + 1))
    
    return [P0, Pr, Q, A, kc]
results = markov_model(1, 2)
print_stats(results)

In [ ]:
import numpy as np
import math
from collections import deque

def erlang_c(p, n, m):
    """Формула Эрланга C для расчета вероятности ожидания (с учетом очереди m)."""
    numerator = (p  n) / math.factorial(n) * (p  m) / math.factorial(m)
    denominator = sum((p ** k) / math.factorial(k) for k in range(n + m + 1))
    return numerator / denominator

def erlang_p0(p, n, m):
    """Вероятность простоя P0."""
    return 1 / sum((p ** k) / math.factorial(k) for k in range(n + m + 1))

def simulate_queue(lambda_rate, mu, channels, queue_size, sim_time=10000):
    """Имитационная модель работы системы."""
    busy, loss, total, queue, time, departures = 0, 0, 0, deque(), 0.0, []
    total_wait_time = 0.0  # Общее время ожидания в очереди
    next_arrival = np.random.exponential(1 / lambda_rate)

    while time < sim_time:
        if departures and departures[0] < next_arrival:
            # Освобождение процессора
            time = departures.pop(0)
            busy -= 1
            
            # Если есть заявки в очереди, берем первую из очереди
            if queue:
                arrival_time = queue.popleft()
                wait_time = time - arrival_time
                total_wait_time += wait_time
                busy += 1
                departures.append(time + np.random.exponential(1 / mu))
                departures.sort()
        else:
            # Новая заявка поступает
            time = next_arrival
            total += 1

            if busy < channels:
                busy += 1
                departures.append(time + np.random.exponential(1 / mu))
                departures.sort()
            elif len(queue) < queue_size:
                queue.append(time)
            else:
                loss += 1

            next_arrival = time + np.random.exponential(1 / lambda_rate)

    p_loss = loss / total if total > 0 else 0
    avg_wait_time = total_wait_time / (total - loss) if (total - loss) > 0 else 0
    avg_system_time = avg_wait_time + (1 / mu)
    return p_loss, avg_wait_time, avg_system_time

# Параметры системы
lambda_fixed, mu, channels, queue_size, sim_time = 0.83, 0.6, 3, 2, 10000
p = lambda_fixed / mu  # Коэффициент загрузки

# Теоретические расчеты
p0 = erlang_p0(p, channels, queue_size)
p_loss = erlang_c(p, channels, queue_size)
q = 1 - p_loss  # Относительная пропускная способность
a = lambda_fixed * q  # Абсолютная пропускная способность
n_busy = q * p  # Среднее число занятых каналов

# Среднее число программ в очереди
n_queue = (p_loss * p) / (1 - p_loss)

# Среднее время нахождения в очереди
w_queue = n_queue / (lambda_fixed * (1 - p_loss))

# Среднее время нахождения в системе
w_system = w_queue + (1 / mu)

# Симуляция
sim_p_loss, sim_w_queue, sim_w_system = simulate_queue(lambda_fixed, mu, channels, queue_size, sim_time)
sim_q = 1 - sim_p_loss
sim_a = lambda_fixed * sim_q
sim_n_busy = sim_a / mu
sim_n_queue = (sim_p_loss * p) / (1 - sim_p_loss)

# Вывод результатов
print(f"\n=== Теоретическая модель ===")
print(f"Вероятность простоя P0: {p0:.5f}")
print(f"Вероятность отказа Pотк: {p_loss:.5f}")
print(f"Относительная пропускная способность Q: {q:.5f}")
print(f"Абсолютная пропускная способность A: {a:.5f}")
print(f"Среднее число занятых каналов Nзан: {n_busy:.5f}")
print(f"Среднее число программ в очереди Nоч: {n_queue:.5f}")
print(f"Среднее время нахождения в очереди Wоч: {w_queue:.5f}")
print(f"Среднее время нахождения в системе Wс: {w_system:.5f}")

print(f"\n=== Имитационная модель ===")
print(f"Вероятность отказа (симуляция) Pотк: {sim_p_loss:.5f}")
print(f"Относительная пропускная способность Q: {sim_q:.5f}")
print(f"Абсолютная пропускная способность A: {sim_a:.5f}")
print(f"Среднее число занятых каналов Nзан: {sim_n_busy:.5f}")
print(f"Среднее число программ в очереди Nоч: {sim_n_queue:.5f}")
print(f"Среднее время нахождения в очереди Wоч: {sim_w_queue:.5f}")
print(f"Среднее время нахождения в системе Wс: {sim_w_system:.5f}")